# Masalah: Memperkirakan Penundaan Pesawat

Tujuan dari notebook ini adalah:
- Proses dan membuat set data dari file .zip yang diunduh
- Melakukan analisis data eksploratif (EDA)
- Menetapkan model dasar
- Pindah dari model sederhana ke model ansambel
- Lakukan optimasi hyperparameter
- Periksa pentingnya fitur


## Pengantar skenario bisnis

Anda bekerja untuk situs web pemesanan perjalanan yang ingin meningkatkan pengalaman pelanggan untuk penerbangan yang tertunda. Perusahaan ingin membuat fitur untuk memberi tahu pelanggan bahwa penerbangan akan tertunda karena cuaca saat mereka memesan penerbangan ke atau dari bandara tersibuk untuk perjalanan domestik di AS. 

Anda ditugaskan untuk memecahkan sebagian dari masalah ini dengan menggunakan machine learning (ML) untuk mengidentifikasi apakah penerbangan akan tertunda karena cuaca. Anda telah diberi akses ke set data tentang performa tepat waktu penerbangan domestik yang dioperasikan oleh maskapai penerbangan besar. Anda dapat menggunakan data ini untuk melatih model ML guna memprediksi apakah penerbangan akan ditunda untuk bandara tersibuk.


## Tentang set data ini

Set data ini berisi waktu keberangkatan dan kedatangan terjadwal dan aktual yang dilaporkan oleh maskapai penerbangan AS bersertifikat yang menyumbang setidaknya 1 persen dari pendapatan penumpang terjadwal domestik. Data tersebut dikumpulkan oleh A.S. Kantor Informasi Maskapai Penerbangan, Biro Statistik Transportasi (BTS). Set data berisi tanggal, waktu, asal, tujuan, maskapai penerbangan, jarak, dan status penundaan penerbangan untuk penerbangan antara 2013 dan 2018.


### Fitur

Untuk informasi selengkapnya tentang fitur dalam set data, lihat [Fitur set data penundaan on-time](https://www.transtats.bts.gov/Fields.asp).

### Atribusi set data  
Situs web: https://www.transtats.bts.gov/

Set data yang digunakan di lab ini disusun oleh A.S. Kantor Informasi Maskapai Penerbangan, Biro Statistik Transportasi (BTS), Data Performa On-Time Maskapai Penerbangan, tersedia di https://www.transtats.bts.gov/DatabaseInfo.asp?DB_ID=120&amp;DB_URL=Mode_ID=1&amp;Mode_Desc=Aviation&amp;Subject_ID2=0.

# Langkah 1: Formulasi masalah dan pengumpulan data

Mulailah proyek ini dengan menulis beberapa kalimat yang merangkum masalah bisnis dan tujuan bisnis yang ingin Anda capai dalam skenario ini. Anda dapat menuliskan ide Anda di dalam bagian berikut. Sertakan metrik bisnis yang menurut Anda perlu dicapai oleh tim Anda. Setelah Anda menentukan informasi itu, tulis pernyataan masalah ML. Terakhir, tambahkan satu atau dua komentar tentang jenis ML yang diwakili aktivitas ini. 

#### <span style="color: blue;">Presentasi proyek: Sertakan ringkasan detail ini dalam presentasi proyek Anda. </span>

### 1. Tentukan apakah dan mengapa ML adalah solusi yang tepat untuk menyebarkan skenario ini.

In [ ]:
# Write your answer here

### 2. Rumuskan masalah bisnis, metrik sukses, dan output ML yang diinginkan.

In [ ]:
# Write your answer here

### 3. Identifikasi jenis masalah ML yang sedang Anda kerjakan.

In [ ]:
# Write your answer here

### 4. Analisis kesesuaian data yang sedang Anda kerjakan.

In [ ]:
# Write your answer here

### Penyiapan

Sekarang setelah Anda memutuskan hal yang akan Anda perhatikan, Anda akan menyiapkan lab ini sehingga Anda dapat mulai memecahkan masalah.

**Catatan:** Notebook ini dibuat dan diuji pada instans notebook `ml.m4.xlarge` dengan penyimpanan 25 GB. 

In [ ]:
import os
from pathlib2 import Path
from zipfile import ZipFile
import time

import pandas as pd
import numpy as np
import subprocess

import matplotlib.pyplot as plt
import seaborn as sns

sns.set()
instance_type='ml.m4.xlarge'

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# Langkah 2: Pra-pemrosesan dan visualisasi data  
Dalam fase pra-pemrosesan data ini, Anda menjelajahi dan memvisualisasikan data Anda untuk lebih memahaminya. Pertama, impor pustaka yang diperlukan dan baca data ke dalam pandas DataFrame. Setelah Anda mengimpor data, lihat apa saja set data. Carilah bentuk set data dan lihat apa saja kolom Anda dan jenis kolom yang akan Anda kerjakan (numerik, kategoris). Pertimbangkan untuk melakukan statistik dasar pada fitur untuk memahami sarana dan rentang fitur. Periksa kolom target Anda dengan cermat, dan tentukan distribusinya.


### Pertanyaan khusus untuk dipertimbangkan

Sepanjang bagian lab ini, pikirkan pertanyaan berikut:

1. Apa yang dapat Anda simpulkan dari statistik dasar yang Anda jalankan pada fitur? 
2. Apa yang bisa Anda simpulkan dari distribusi kelas target?
3. Apakah ada hal lain yang dapat Anda simpulkan dengan melihat-lihat isi data tersebut?

#### <span style="color: blue;">Presentasi proyek: Sertakan ringkasan jawaban Anda atas pertanyaan-pertanyaan ini (dan pertanyaan serupa lainnya) dalam presentasi proyek Anda.</span>

Mulailah dengan membawa set data dari bucket Amazon Simple Storage Service (Amazon S3) publik ke lingkungan notebook ini.

In [ ]:
# download the files

zip_path = '/home/ec2-user/SageMaker/project/data/FlightDelays/'
base_path = '/home/ec2-user/SageMaker/project/data/FlightDelays/'
csv_base_path = '/home/ec2-user/SageMaker/project/data/csvFlightDelays/'

!mkdir -p {zip_path}
!mkdir -p {csv_base_path}
!aws s3 cp s3://aws-tc-largeobjects/CUR-TF-200-ACMLFO-1/flight_delay_project/data/ {zip_path} --recursive


In [ ]:
zip_files = [str(file) for file in list(Path(base_path).iterdir()) if '.zip' in str(file)]
len(zip_files)

Ekstrak file comma-separated values (CSV) dari file.zip.

In [ ]:
def zip2csv(zipFile_name , file_path):
    """
    Extract csv from zip files
    zipFile_name: name of the zip file
    file_path : name of the folder to store csv
    """

    try:
        with ZipFile(zipFile_name, 'r') as z: 
            print(f'Extracting {zipFile_name} ') 
            z.extractall(path=file_path) 
    except:
        print(f'zip2csv failed for {zipFile_name}')

for file in zip_files:
    zip2csv(file, csv_base_path)

print("Files Extracted")

In [ ]:
csv_files = [str(file) for file in list(Path(csv_base_path).iterdir()) if '.csv' in str(file)]
len(csv_files)

Sebelum Anda memuat file CSV, baca file HTML dari folder yang diekstrak. File HTML ini mencakup latar belakang dan informasi lebih lanjut tentang fitur yang disertakan dalam set data.

In [ ]:
from IPython.display import IFrame

IFrame(src=os.path.relpath(f"{csv_base_path}readme.html"), width=1000, height=600)

#### Muat sampel file CSV

Sebelum Anda menggabungkan semua file CSV, periksa data dari satu file CSV. Dengan menggunakan pandas, bacalah file `On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2018_9.csv` terlebih dahulu. Anda dapat menggunakan fungsi bawaan `read_csv` dalam Python (dokumentasi [pandas.read_csv](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html)).

In [ ]:
df_temp = pd.read_csv(f"{csv_base_path}On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2018_9.csv")

**Pertanyaan**: Cetak panjang baris dan kolom dalam set data, dan cetak nama kolom.

**Petunjuk**: Untuk melihat baris dan kolom dari DataFrame, gunakan fungsi `<DataFrame>.shape` function. To view the column names, use the `<DataFrame>.columns`.

In [ ]:
df_shape = # **ENTER YOUR CODE HERE**
print(f'Rows and columns in one CSV file is {df_shape}')

**Pertanyaan**: Cetak 10 baris pertama dari set data.  

**Petunjuk**: Untuk mencetak `x` jumlah baris, gunakan `head(x)` fungsi bawaan di pandas.

In [ ]:
# Enter your code here

**Pertanyaan**: Cetak semua kolom dalam set data. Untuk melihat nama kolom, gunakan `<DataFrame>.columns`.

In [ ]:
print(f'The column names are :')
print('#########')
for col in <CODE>:# **ENTER YOUR CODE HERE**
    print(col)

**Pertanyaan**: Cetak semua kolom di set data yang berisi kata *Del*. Ini akan membantu Anda melihat berapa banyak kolom yang memiliki *data penundaan* di dalamnya.

**Petunjuk**: Untuk menyertakan nilai-nilai yang memenuhi kriteria pernyataan `if` tertentu, Anda dapat menggunakan pemahaman daftar Python.

Sebagai contoh: `[x for x in [1,2,3,4,5] if x > 2]`  

**Petunjuk**: Untuk memeriksa apakah nilai terdapat dalam daftar, Anda dapat menggunakan kata kunci `in` ([Python dalam dokumentasi Kata Kunci] (https://www.w3schools.com/python/ref_keyword_in.asp)). 

Sebagai contoh: `5 in [1,2,3,4,5]`

In [ ]:
# Enter your code here

Berikut adalah beberapa pertanyaan lain yang akan membantu Anda mempelajari set data Anda selengkapnya.

**Pertanyaan**   

1. Berapa banyak baris dan kolom yang dimiliki set data?   
2. Berapa tahun yang termasuk dalam set data?   
3. Apa rentang tanggal untuk set data?   
4. Maskapai penerbangan mana yang termasuk dalam set data?   
5. Bandara asal dan tujuan mana yang tercakup?

**Petunjuk**
- Untuk menunjukkan dimensi DataFrame, gunakan `df_temp.shape`.
- Untuk merujuk ke kolom tertentu, gunakan `df_temp.columnName` (misalnya, `df_temp.CarrierDelay`).
- Untuk mendapatkan nilai yang unik untuk kolom, gunakan `df_temp.column.unique()` (misalnya `df_temp.Year.unique()`).

In [ ]:
print("The #rows and #columns are ", <CODE> , " and ", <CODE>)
print("The years in this dataset are: ", <CODE>)
print("The months covered in this dataset are: ", <CODE>)
print("The date range for data is :" , min(<CODE>), " to ", max(<CODE>))
print("The airlines covered in this dataset are: ", list(<CODE>))
print("The Origin airports covered are: ", list(<CODE>))
print("The Destination airports covered are: ", list(<CODE>))

**Pertanyaan**: Berapa jumlah semua bandara keberangkatan dan tujuan?

**Petunjuk**: Untuk menemukan nilai untuk setiap bandara dengan menggunakan kolom **Asal** dan **Tujuan**, Anda dapat menggunakan `values_count` fungsi dalam pandas ([dokumentasi pandas.Series.value_counts] (https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.value_counts.html)).

In [ ]:
counts = pd.DataFrame({'Origin':<CODE>, 'Destination':<CODE>})
counts

**Pertanyaan**: Cetak 15 bandara asal dan tujuan teratas berdasarkan jumlah penerbangan dalam set data.

**Petunjuk**: Anda dapat menggunakan `sort_values` fungsi dalam pandas ([dokumentasi pandas.DataFrame.sort_values] (https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.sort_values.html)).

In [ ]:
counts.sort_values(by=<CODE>,ascending=False).head(15) # Enter your code here

**Mengingat semua informasi tentang perjalanan penerbangan, dapatkah Anda memprediksi apakah suatu penerbangan akan tertunda?**

Kolom **ArrDel15** adalah variabel indikator yang mengambil nilai *1* ketika penundaan lebih dari 15 menit. Jika tidak, dibutuhkan nilai *0*.

Anda dapat menggunakan ini sebagai kolom target untuk masalah klasifikasi.

Sekarang, asumsikan Anda bepergian dari San Francisco ke Los Angeles dalam perjalanan kerja. Anda ingin mengelola reservasi Anda dengan lebih baik di Los Angeles. Dengan demikian, Anda ingin tahu apakah penerbangan Anda akan tertunda, dengan mengandalkan seperangkat fitur. Berapa banyak fitur dari set data ini yang perlu Anda ketahui sebelum penerbangan Anda?

Kolom seperti `DepDelay`,`ArrDelay`, `CarrierDelay`, `WeatherDelay`, `NASDelay`,`SecurityDelay`, `LateAircraftDelay`, dan `DivArrDelay` berisi informasi tentang penundaan. Tapi penundaan ini bisa terjadi di lokasi keberangkatan atau tujuan. Jika terjadi penundaan mendadak akibat cuaca 10 menit sebelum mendarat, data ini tidak akan membantu untuk mengelola reservasi Los Angeles Anda.

Jadi untuk menyederhanakan pernyataan masalah, manfaatkan kolom berikut untuk memprediksi kedatangan yang tertunda:<br>

`Year`, `Quarter`, `Month`, `DayofMonth`, `DayOfWeek`, `FlightDate`, `Reporting_Airline`, `Origin`, `OriginState`, `Dest`, `DestState`, `CRSDepTime`, `DepDelayMinutes`, `DepartureDelayGroups`, `Cancelled`, `Diverted`, `Distance`, `DistanceGroup`, `ArrDelay`, `ArrDelayMinutes`, `ArrDel15`, `AirTime`

Anda juga akan memfilter bandara sumber dan tujuan agar dapat melihat:
- Bandara teratas: ATL, ORD, DFW, DEN, CLT, LAX, IAH, PHX, SFO
- Lima maskapai penerbangan teratas: UA, OO, WN, AA, DL

Informasi ini akan turut mengurangi ukuran data di seluruh file CSV yang akan digabungkan.

#### Gabungkan semua file CSV
 
Pertama, buat DataFrame kosong yang akan Anda gunakan untuk menyalin DataFrame individu Anda dari setiap file. Kemudian, untuk setiap file dalam daftar `csv_files`:

1. Baca file CSV ke dalam dataframe 
2. Filter kolom berdasarkan variabel `filter_cols`

```
        columns = ['col1', 'col2']
        df_filter = df[columns]
```

3. Simpan hanya `subset_vals` di masing-masing `subset_cols`. Untuk memeriksa apakah `val` berada di kolom DataFrame, gunakan fungsi `isin` dalam pandas ([dokumentasi pandas.DataFram.isin] (https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.isin.html)). Kemudian, pilih baris yang menyertakannya.

```
        df_eg[df_eg['col1'].isin('5')]
```

4. Gabungkan DataFrame dengan DataFrame kosong 

In [ ]:
def combine_csv(csv_files, filter_cols, subset_cols, subset_vals, file_name):

    """
    Combine csv files into one Data Frame
    csv_files: list of csv file paths
    filter_cols: list of columns to filter
    subset_cols: list of columns to subset rows
    subset_vals: list of list of values to subset rows
    """

    df = pd.DataFrame()
    
    for file in csv_files:
        df_temp = pd.read_csv(file)
        df_temp = df_temp[filter_cols]
        for col, val in zip(subset_cols,subset_vals):
            df_temp = df_temp[df_temp[col].isin(val)]      
        
        df = pd.concat([df, df_temp], axis=0)
      
    df.to_csv(file_name, index=False)
    print(f'Combined csv stored at {file_name}')

In [ ]:
#cols is the list of columns to predict Arrival Delay 
cols = ['Year','Quarter','Month','DayofMonth','DayOfWeek','FlightDate',
        'Reporting_Airline','Origin','OriginState','Dest','DestState',
        'CRSDepTime','Cancelled','Diverted','Distance','DistanceGroup',
        'ArrDelay','ArrDelayMinutes','ArrDel15','AirTime']

subset_cols = ['Origin', 'Dest', 'Reporting_Airline']

# subset_vals is a list collection of the top origin and destination airports and top 5 airlines
subset_vals = [['ATL', 'ORD', 'DFW', 'DEN', 'CLT', 'LAX', 'IAH', 'PHX', 'SFO'], 
               ['ATL', 'ORD', 'DFW', 'DEN', 'CLT', 'LAX', 'IAH', 'PHX', 'SFO'], 
               ['UA', 'OO', 'WN', 'AA', 'DL']]

Gunakan fungsi sebelumnya untuk menggabungkan semua file yang berbeda menjadi satu file yang dapat Anda baca dengan mudah. 

**Catatan**: Proses ini dapat selesai dalam waktu 5-7 menit.

In [ ]:
start = time.time()
combined_csv_filename = f"{base_path}combined_files.csv"
combine_csv(csv_files, cols, subset_cols, subset_vals, combined_csv_filename)
print(f'CSVs merged in {round((time.time() - start)/60,2)} minutes')

#### Muat set data

Memuat set data gabungan.

In [ ]:
data = pd.read_csv(combined_csv_filename)

Cetak lima catatan pertama.

In [ ]:
# Enter your code here 

Berikut adalah beberapa pertanyaan lain yang akan membantu Anda mempelajari set data Anda selengkapnya.

**Pertanyaan**   

1. Berapa banyak baris dan kolom yang dimiliki set data?   
2. Berapa tahun yang termasuk dalam set data?   
3. Apa rentang tanggal untuk set data?   
4. Maskapai penerbangan mana yang termasuk dalam set data?   
5. Bandara asal dan tujuan mana yang tercakup?

In [ ]:
print("The #rows and #columns are ", <CODE> , " and ", <CODE>)
print("The years in this dataset are: ", list(<CODE>))
print("The months covered in this dataset are: ", sorted(list(<CODE>)))
print("The date range for data is :" , min(<CODE>), " to ", max(<CODE>))
print("The airlines covered in this dataset are: ", list(<CODE>))
print("The Origin airports covered are: ", list(<CODE>))
print("The Destination airports covered are: ", list(<CODE>))

Tentukan kolom target Anda: **is_delay** (*1* berarti waktu kedatangan tertunda lebih dari 15 menit, dan *0* berarti semua kasus lainnya). Untuk mengubah nama kolom dari **ArrDel15** ke *is_delay*, gunakan metode `rename`.

**Petunjuk**: Anda dapat menggunakan `rename` fungsi dalam pandas ([dokumentasi pandas.DataFrame.rename] (https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rename.html)).

Sebagai contoh:
```
data.rename(columns={'col1':'column1'}, inplace=True)
```

In [ ]:
data.rename(columns=<CODE>, inplace=True) # Enter your code here

Cari nulls di seluruh kolom. Anda dapat menggunakan fungsi `isnull()` ([dokumentasi pandas.isnull](https://pandas.pydata.org/pandas-docs/version/0.17.0/generated/pandas.isnull.html)).

**Petunjuk**: `isnull()` mendeteksi apakah nilai tertentu adalah nol atau bukan. Ia mengembalikan boolean (*Betul* atau *Salah*) di tempatnya. Untuk menghitung seluruh jumlah kolom, gunakan fungsi `sum(axis=0)` (misalnya, `df.isnull().sum(axis=0)`).

In [ ]:
# Enter your code here

Detail keterlambatan kedatangan dan airtime hilang untuk 22.540 dari 1.658.130 baris, yaitu 1,3 persen. Anda dapat menghapus atau menghubungkan baris ini. Dokumentasi tidak menyebutkan informasi tentang baris yang hilang.


In [ ]:
### Remove null columns
data = data[~data.is_delay.isnull()]
data.isnull().sum(axis = 0)

Atur jam harian dalam format waktu 24 jam dari CRSDepTime.

In [ ]:
data['DepHourofDay'] = (data['CRSDepTime']//100)

## **Pernyataan masalah ML**
- Dengan set fitur, dapat Anda memprediksi apakah penerbangan akan tertunda lebih dari 15 menit?
- Karena variabel target hanya membutuhkan nilai *0* atau *1*, Anda dapat menggunakan algoritme klasifikasi. 

Sebelum memulai pemodelan, sebaiknya perhatikan distribusi fitur, korelasi, dan lain-lain.
- ini akan memberikan gambaran dari setiap non-linearitas atau pola dalam data
    - Model linier: Tambahkan fitur daya, eksponensial, atau interaksi
    - Coba model non-linear
- Ketidakseimbangan data 
    - Pilih metrik yang tidak akan memberikan performa model yang bias (akurasi versus area di bawah kurva, atau AUC)
    - Gunakan fungsi kerugian tertimbang atau kustom
- Data hilang
    - Lakukan imputasi berdasarkan statistik sederhana -- mean, median, modus (variabel numerik), frequent class (variabel kategoris)
    - Imputasi berbasis klaster (k-nearest neighbor, atau KNN, untuk memprediksi nilai kolom)
    - Letakkan kolom

### Eksplorasi data

Periksa kelas *penundaan* versus *tanpa penundaan*.


In [ ]:
(data.groupby('is_delay').size()/len(data) ).plot(kind='bar')# Enter your code here
plt.ylabel('Frequency')
plt.title('Distribution of classes')
plt.show()

**Pertanyaan**: Apa yang dapat Anda simpulkan dari plot batang tentang rasio *penundaan* versus *tanpa penundaan*?

In [ ]:
# Enter your answer here

Jalankan dua sel berikut dan jawab pertanyaannya.

In [ ]:
viz_columns = ['Month', 'DepHourofDay', 'DayOfWeek', 'Reporting_Airline', 'Origin', 'Dest']
fig, axes = plt.subplots(3, 2, figsize=(20,20), squeeze=False)
# fig.autofmt_xdate(rotation=90)

for idx, column in enumerate(viz_columns):
    ax = axes[idx//2, idx%2]
    temp = data.groupby(column)['is_delay'].value_counts(normalize=True).rename('percentage').\
    mul(100).reset_index().sort_values(column)
    sns.barplot(x=column, y="percentage", hue="is_delay", data=temp, ax=ax)
    plt.ylabel('% delay/no-delay')
    

plt.show()

In [ ]:
sns.lmplot( x="is_delay", y="Distance", data=data, fit_reg=False, hue='is_delay', legend=False)
plt.legend(loc='center')
plt.xlabel('is_delay')
plt.ylabel('Distance')
plt.show()

**Pertanyaan**

Dengan menggunakan data dari grafik sebelumnya, jawablah pertanyaan-pertanyaan ini:

- Bulan apa yang mengalami paling banyak penundaan?
- Dalam satu hari, kapan penundaan paling sering terjadi?
- Hari apa dalam seminggu yang mengalami paling banyak penundaan?
- Maskapai mana yang memiliki penundaan paling banyak?
- Bandara keberangkatan dan tujuan mana yang paling banyak mengalami penundaan?
- Apakah jarak penerbangan merupakan faktor penundaan?

In [ ]:
# Enter your answers here

### Fitur

Lihatlah semua kolom dan jenis spesifik apa yang dimiliki kolom tersebut.

In [ ]:
data.columns

In [ ]:
data.dtypes

Memfilter kolom yang diperlukan:
- *Tanggal* tidak lagi diperlukan, karena Anda memiliki *Tahun*, *Kuartal*, *Bulan*, *HariBulan*, dan *HariMinggu* untuk menjelaskan tanggalnya.
- Gunakan kode *Asal* dan *Tujuan* alih-alih *NegaraAsal* dan *NegaraTujuan*.
- Karena Anda hanya mengklasifikasikan apakah penerbangan tertunda atau tidak, Anda tidak memerlukan *TotalMenitTertunda*, *MenitBerangkatTertunda*, dan *MenitKedatanganTertunda*.

Perlakukan *JamKeberangkatanHari* sebagai variabel kategoris karena tidak memiliki hubungan kuantitatif dengan target.
- Jika Anda perlu melakukan pengkodean one-hot pada variabel ini, 23 kolom lebih akan dihasilkan.
- Alternatif lain untuk menangani variabel kategoris mencakup misalnya pengkodean hash, pengkodean rata-rata teratur, dan buket nilai.
- Dalam hal ini, Anda hanya perlu memisahkannya menjadi bucket.

Untuk mengubah jenis kolom menjadi kategori, gunakan fungsi `astype` ([dokumentasi pandas.DataFrame.astype](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.astype.html)).

In [ ]:
data_orig = data.copy()
data = data[[ 'is_delay', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 
       'Reporting_Airline', 'Origin', 'Dest','Distance','DepHourofDay']]
categorical_columns  = ['Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 
       'Reporting_Airline', 'Origin', 'Dest', 'DepHourofDay']
for c in categorical_columns:
    data[c] = data[c].astype('category')

Untuk menggunakan pengkodean one-hot, gunakan fungsi `get_dummies` dalam pandas untuk kolom kategoris yang Anda pilih. Kemudian, Anda dapat menggabungkan fitur-fitur yang dihasilkan untuk set data asli Anda dengan menggunakan fungsi `concat` dalam pandas. Untuk mengkodekan variabel kategoris, Anda juga dapat menggunakan *pengkodean contoh* dengan menggunakan kata kunci `drop_first=True`. Untuk informasi lebih lanjut tentang pengkodean dummy, lihat [variabel Dummy (statistik)] (https://en.wikiversity.org/wiki/Dummy_variable_(statistics)).

Sebagai contoh:
```
pd.get_dummies(df[['column1','columns2']], drop_first=True)
```

In [ ]:
data_dummies = pd.get_dummies(<CODE>, drop_first=True) # Enter your code here
data = pd.concat([<CODE>, <CODE>], axis = 1)
data.drop(categorical_columns,axis=1, inplace=True)

Periksa panjang set data dan kolom baru.

**Petunjuk**: Gunakan `shape` dan properti `columns`.

In [ ]:
# Enter your code here

In [ ]:
# Enter your code here

Anda sekarang siap untuk melatih model. Sebelum Anda memisahkan data, ganti nama kolom **is_delay** menjadi *target*.

**Petunjuk**: Anda dapat menggunakan `rename` fungsi dalam pandas ([dokumentasi pandas.DataFrame.rename] (https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rename.html)).

In [ ]:
data.rename(columns = {<CODE>:<CODE>}, inplace=True )# Enter your code here

## <span style="color:red"> Akhir dari Langkah 2 </span>

Simpan file proyek ke komputer lokal Anda. Ikuti langkah-langkah berikut:

1. Di file explorer sebelah kiri, klik kanan buku catatan yang sedang Anda kerjakan. 

2. Pilih **Unduh**, dan simpan file secara lokal.  

Tindakan ini mengunduh buku catatan saat ini ke folder unduhan default di komputer Anda.

# Langkah 3: Pelatihan dan evaluasi model

Anda harus menyertakan beberapa langkah awal ketika Anda mengkonversi set data dari DataFrame ke format yang dapat digunakan oleh algoritme machine learning. Untuk Amazon SageMaker, Anda harus melakukan langkah-langkah berikut:

1. Memisahkan data menjadi `train_data`, `validation_data`, dan `test_data` dengan menggunakan `sklearn.model_selection.train_test_split`.  

2. Konversikan set data ke format file yang sesuai yang dapat digunakan oleh tugas pelatihan Amazon SageMaker. Ini bisa berupa file CSV atau protobuf catatan. Untuk informasi lebih lanjut, lihat [Format Data Umum untuk Pelatihan] (https://docs.aws.amazon.com/sagemaker/latest/dg/cdf-training.html).  

3. Unggah data ke bucket S3 Anda. Jika Anda belum membuatnya sebelumnya, lihat [Buat Bucket] (https://docs.aws.amazon.com/AmazonS3/latest/gsg/CreatingABucket.html).  

Gunakan sel berikut untuk menyelesaikan langkah-langkah ini. Masukkan dan hapus sel jika diperlukan.

#### <span style="color: blue;">Presentasi proyek: Dalam presentasi proyek Anda, tuliskan keputusan kunci yang Anda buat dalam fase ini. </span>

### Pemisahan uji-latihan

In [ ]:
from sklearn.model_selection import train_test_split
def split_data(data):
    train, test_and_validate = train_test_split(data, test_size=0.2, random_state=42, stratify=data['target'])
    test, validate = train_test_split(test_and_validate, test_size=0.5, random_state=42, stratify=test_and_validate['target'])
    return train, validate, test

In [ ]:
train, validate, test = split_data(data)
print(train['target'].value_counts())
print(test['target'].value_counts())
print(validate['target'].value_counts())

**Jawaban sampel**
```
0.0    1033570
1.0     274902
Name: target, dtype: int64
0.0    129076
1.0     34483
Name: target, dtype: int64
0.0    129612
1.0     33947
Name: target, dtype: int64
```

### Model klasifikasi dasar

In [ ]:
import sagemaker
from sagemaker.serializers import CSVSerializer
from sagemaker.amazon.amazon_estimator import RecordSet
import boto3

# Instantiate the LinearLearner estimator object with 1 ml.m4.xlarge
classifier_estimator = sagemaker.LinearLearner(role=sagemaker.get_execution_role(),
                                               instance_count=<CODE>,
                                               instance_type=<CODE>,
                                               predictor_type=<CODE>,
                                               binary_classifier_model_selection_criteria=<CODE>)

### Kode sampel
```
num_classes = len(pd.unique(train_labels))
classifier_estimator = sagemaker.LinearLearner(role=sagemaker.get_execution_role(),
                                              instance_count=1,
                                              instance_type='ml.m4.xlarge',
                                              predictor_type='binary_classifier',
                                              binary_classifier_model_selection_criteria = 'cross_entropy_loss')
                                              
```

Peserta linear menerima data pelatihan dalam jenis konten protobuf atau CSV. Peserta linear ini juga menerima permintaan kesimpulan dalam jenis konten protobuf, CSV, atau JavaScript Object Notation (JSON). Data pelatihan memiliki fitur dan label ground-truth, tetapi data dalam permintaan inferensi hanya memiliki fitur.

Dalam alur produksi, AWS merekomendasikan perubahan data ke format protobuf Amazon SageMaker dan menyimpannya di Amazon S3. Untuk aktif dan berjalan dengan cepat, AWS menyediakan `record_set` operasi untuk mengkonversi dan mengunggah set data jika ukurannya cukup kecil sehingga dapat dimuat dalam memori lokal. Operasi ini menerima array NumPy seperti yang Anda miliki, sehingga Anda akan menggunakannya untuk langkah ini. Objek `RecordSet` akan melacak lokasi Amazon S3 sementara data Anda. Buat latihan, validasi, dan catatan pengujian dengan menggunakan fungsi `estimator.record_set`. Kemudian, mulai tugas pelatihan Anda dengan menggunakan fungsi `estimator.fit`.

In [ ]:
### Create train, validate, and test records
train_records = classifier_estimator.record_set(train.values[:, 1:].astype(np.float32), train.values[:, 0].astype(np.float32), channel='train')
val_records = classifier_estimator.record_set(validate.values[:, 1:].astype(np.float32), validate.values[:, 0].astype(np.float32), channel='validation')
test_records = classifier_estimator.record_set(test.values[:, 1:].astype(np.float32), test.values[:, 0].astype(np.float32), channel='test')

Sekarang, latih model Anda pada set data yang baru saja Anda unggah.

### Kode sampel
```
linear.fit([train_records,val_records,test_records])
```

In [ ]:
### Fit the classifier
# Enter your code here

## Evaluasi model
Pada bagian ini, Anda akan mengevaluasi model terlatih Anda. 

Pertama, periksa metrik untuk tugas pelatihan:

In [ ]:
sagemaker.analytics.TrainingJobAnalytics(classifier_estimator._current_job_name, 
                                         metric_names = ['test:objective_loss', 
                                                         'test:binary_f_beta',
                                                         'test:precision',
                                                         'test:recall']
                                        ).dataframe()

Selanjutnya, atur beberapa fungsi yang akan membantu memuat data pengujian ke Amazon S3 dan lakukan prediksi dengan menggunakan fungsi prediksi batch. Menggunakan prediksi batch akan membantu mengurangi biaya karena instans hanya akan berjalan ketika prediksi dilakukan pada data pengujian yang disediakan.

**Catatan: ** Ganti `<LabBucketName>` dengan nama bucket lab yang dibuat selama pengaturan lab.

In [ ]:
import io
#bucket='<LabBucketName>'
prefix='flight-linear'
train_file='flight_train.csv'
test_file='flight_test.csv'
validate_file='flight_validate.csv'
whole_file='flight.csv'
s3_resource = boto3.Session().resource('s3')

def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, header=False, index=False )
    s3_resource.Bucket(bucket).Object(os.path.join(prefix, folder, filename)).put(Body=csv_buffer.getvalue())

In [ ]:
def batch_linear_predict(test_data, estimator):
    batch_X = test_data.iloc[:,1:];
    batch_X_file='batch-in.csv'
    upload_s3_csv(batch_X_file, 'batch-in', batch_X)

    batch_output = "s3://{}/{}/batch-out/".format(bucket,prefix)
    batch_input = "s3://{}/{}/batch-in/{}".format(bucket,prefix,batch_X_file)

    classifier_transformer = estimator.transformer(instance_count=1,
                                           instance_type='ml.m4.xlarge',
                                           strategy='MultiRecord',
                                           assemble_with='Line',
                                           output_path=batch_output)

    classifier_transformer.transform(data=batch_input,
                             data_type='S3Prefix',
                             content_type='text/csv',
                             split_type='Line')
    
    classifier_transformer.wait()

    s3 = boto3.client('s3')
    obj = s3.get_object(Bucket=bucket, Key="{}/batch-out/{}".format(prefix,'batch-in.csv.out'))
    target_predicted_df = pd.read_json(io.BytesIO(obj['Body'].read()),orient="records",lines=True)
    return test_data.iloc[:,0], target_predicted_df.iloc[:,0]


Untuk menjalankan prediksi pada set data pengujian, jalankan fungsi `batch_linear_predict` (yang didefinisikan sebelumnya) pada set data pengujian Anda.


In [ ]:
test_labels, target_predicted = batch_linear_predict(test, classifier_estimator)

Untuk melihat plot matriks kekeliruan, dan berbagai metrik penilaian, buat beberapa fungsi:

In [ ]:
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(test_labels, target_predicted):
    matrix = confusion_matrix(test_labels, target_predicted)
    df_confusion = pd.DataFrame(matrix)
    colormap = sns.color_palette("BrBG", 10)
    sns.heatmap(df_confusion, annot=True, fmt='.2f', cbar=None, cmap=colormap)
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.ylabel("True Class")
    plt.xlabel("Predicted Class")
    plt.show()
    

In [ ]:
from sklearn import metrics

def plot_roc(test_labels, target_predicted):
    TN, FP, FN, TP = confusion_matrix(test_labels, target_predicted).ravel()
    # Sensitivity, hit rate, recall, or true positive rate
    Sensitivity  = float(TP)/(TP+FN)*100
    # Specificity or true negative rate
    Specificity  = float(TN)/(TN+FP)*100
    # Precision or positive predictive value
    Precision = float(TP)/(TP+FP)*100
    # Negative predictive value
    NPV = float(TN)/(TN+FN)*100
    # Fall out or false positive rate
    FPR = float(FP)/(FP+TN)*100
    # False negative rate
    FNR = float(FN)/(TP+FN)*100
    # False discovery rate
    FDR = float(FP)/(TP+FP)*100
    # Overall accuracy
    ACC = float(TP+TN)/(TP+FP+FN+TN)*100

    print("Sensitivity or TPR: ", Sensitivity, "%") 
    print( "Specificity or TNR: ",Specificity, "%") 
    print("Precision: ",Precision, "%") 
    print("Negative Predictive Value: ",NPV, "%") 
    print( "False Positive Rate: ",FPR,"%")
    print("False Negative Rate: ",FNR, "%") 
    print("False Discovery Rate: ",FDR, "%" )
    print("Accuracy: ",ACC, "%") 

    test_labels = test.iloc[:,0];
    print("Validation AUC", metrics.roc_auc_score(test_labels, target_predicted) )

    fpr, tpr, thresholds = metrics.roc_curve(test_labels, target_predicted)
    roc_auc = metrics.auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, label='ROC curve (area = %0.2f)' % (roc_auc))
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver operating characteristic')
    plt.legend(loc="lower right")

    # create the axis of thresholds (scores)
    ax2 = plt.gca().twinx()
    ax2.plot(fpr, thresholds, markeredgecolor='r',linestyle='dashed', color='r')
    ax2.set_ylabel('Threshold',color='r')
    ax2.set_ylim([thresholds[-1],thresholds[0]])
    ax2.set_xlim([fpr[0],fpr[-1]])

    print(plt.figure())

Untuk memplot matriks kekeliruan, panggil fungsi `plot_confusion_matrix` pada data `test_labels` dan `target_predicted` dari tugas batch Anda:

In [ ]:
# Enter your code here

Untuk mencetak statistik dan memplot kurva karakteristik operasi penerima (ROC), panggil fungsi `plot_roc` pada data `test_labels` dan `target_predicted` dari tugas batch Anda:

In [ ]:
# Enter your code here

### Pertanyaan utama yang harus dipertimbangkan:

1. Bagaimana performa model Anda pada set pengujian dibandingkan dengan performanya pada set pelatihan? Apa yang bisa Anda simpulkan dari perbandingan ini? 
2. Adakah perbedaan yang jelas antara hasil metrik seperti akurasi, presisi, dan recall ? Jika demikian, mengapa Anda melihat perbedaan-perbedaan itu? 
3. Mengingat situasi dan sasaran bisnis Anda, metrik (atau metrik) mana yang paling penting untuk Anda pertimbangkan? Mengapa?
4. Dari sudut pandang bisnis, apakah hasil untuk metrik (atau metrik) yang Anda anggap sebagai yang paling penting cukup untuk apa yang Anda butuhkan? Jika tidak, hal-hal apa yang mungkin Anda ubah dalam iterasi berikutnya? (Ini akan terjadi di bagian rekayasa fitur, yang selanjutnya.)

Gunakan sel berikut untuk menjawab pertanyaan ini (dan lainnya). Masukkan dan hapus sel jika diperlukan.

#### <span style="color: blue;">Presentasi proyek: Dalam presentasi proyek Anda, tuliskan jawaban Anda atas pertanyaan-pertanyaan ini -- dan pertanyaan serupa lainnya yang mungkin Anda jawab -- di bagian ini. Catat detail utama dan keputusan yang Anda buat. </span>


**Pertanyaan**: Apa yang dapat Anda simpulkan dari matriks kekeliruan?


In [ ]:
# Enter your answer here

## <span style="color:red"> Akhir dari Langkah 3 </span>

Simpan file proyek ke komputer lokal Anda. Ikuti langkah-langkah berikut:

1. Di file explorer sebelah kiri, klik kanan buku catatan yang sedang Anda kerjakan. 

2. Pilih **Unduh**, dan simpan file secara lokal.  

Tindakan ini mengunduh buku catatan saat ini ke folder unduhan default di komputer Anda.

# Iterasi II

# Langkah 4: Rekayasa fitur

Anda sekarang telah melalui satu iterasi pelatihan dan evaluasi terhadap model Anda. Karena hasil pertama yang Anda capai untuk model Anda mungkin tidak cukup untuk memecahkan masalah bisnis Anda, apa yang bisa Anda ubah tentang data Anda untuk mungkin meningkatkan performa model?

### Pertanyaan utama yang harus dipertimbangkan:

1. Bagaimana keseimbangan dua kelas utama Anda (*penundaan* dan *tanpa penundaan*) memengaruhi performa model?
2. Apakah Anda memiliki fitur yang berkorelasi?
3. Pada tahap ini, bisakah Anda melakukan teknik pengurangan fitur yang mungkin memiliki dampak positif pada performa model? 
4. Apakah menurut Anda ada lebih banyak data atau set data yang dapat ditambahkan?
5. Setelah melakukan beberapa rekayasa fitur, bagaimana performa model Anda dibandingkan dengan iterasi pertama?

Gunakan sel-sel berikut untuk melakukan teknik-teknik rekayasa fitur tertentu yang menurut Anda dapat meningkatkan performa model Anda (gunakan pertanyaan sebelumnya sebagai panduan). Masukkan dan hapus sel jika diperlukan.

#### <span style="color: blue;">Presentasi proyek: Dalam presentasi proyek Anda, catat keputusan kunci Anda dan metode yang Anda gunakan di bagian ini. Juga sertakan metrik performa baru yang Anda peroleh setelah mengevaluasi model Anda lagi. </span>

Sebelum Anda mulai, pikirkan mengapa presisi dan recall-nya sekitar 80 persen, dan akurasinya 99 persen.

Tambahkan lebih banyak fitur:

1. Liburan
2. Cuaca

Karena daftar hari libur dari 2014 hingga 2018 diketahui, Anda dapat membuat variabel indikator **is_holiday** untuk menandainya.

Hipotesisnya adalah penundaan pesawat bisa lebih sering selama liburan dibandingkan dengan hari-hari lainnya. Tambahkan variabel boolean `is_holiday` yang mencakup liburan untuk tahun 2014-2018.

In [ ]:
# Source: http://www.calendarpedia.com/holidays/federal-holidays-2014.html

holidays_14 = ['2014-01-01',  '2014-01-20', '2014-02-17', '2014-05-26', '2014-07-04', '2014-09-01', '2014-10-13', '2014-11-11', '2014-11-27', '2014-12-25' ] 
holidays_15 = ['2015-01-01',  '2015-01-19', '2015-02-16', '2015-05-25', '2015-06-03', '2015-07-04', '2015-09-07', '2015-10-12', '2015-11-11', '2015-11-26', '2015-12-25'] 
holidays_16 = ['2016-01-01',  '2016-01-18', '2016-02-15', '2016-05-30', '2016-07-04', '2016-09-05', '2016-10-10', '2016-11-11', '2016-11-24', '2016-12-25', '2016-12-26']
holidays_17 = ['2017-01-02', '2017-01-16', '2017-02-20', '2017-05-29' , '2017-07-04', '2017-09-04' ,'2017-10-09', '2017-11-10', '2017-11-23', '2017-12-25']
holidays_18 = ['2018-01-01', '2018-01-15', '2018-02-19', '2018-05-28' , '2018-07-04', '2018-09-03' ,'2018-10-08', '2018-11-12','2018-11-22', '2018-12-25']
holidays = holidays_14+ holidays_15+ holidays_16 + holidays_17+ holidays_18

### Add indicator variable for holidays
data_orig['is_holiday'] = # Enter your code here 

Data cuaca diambil dari https://www.ncei.noaa.gov/access/services/data/v1?dataset=daily-summaries&amp;stations=USW00023174,USW00012960,USW00003017,USW00094846,USW00013874,USW00023234,USW00003927,USW00023183,USW00013881&amp;dataTypes=AWND,PRCP,SNOW,SNWD,TAVG,TMIN,TMAX&amp;startDate=2014-01-01&amp;endDate=2018-12-31.
<br>

Set data ini memiliki informasi tentang kecepatan angin, curah hujan, salju, dan suhu untuk kota berdasarkan kode bandara mereka.

**Pertanyaan**: Mungkinkah cuaca buruk karena hujan, angin kencang, atau salju menyebabkan penundaan pesawat? Sekarang Anda akan memeriksanya.

In [ ]:
!aws s3 cp s3://aws-tc-largeobjects/CUR-TF-200-ACMLFO-1/flight_delay_project/data2/daily-summaries.csv /home/ec2-user/SageMaker/project/data/
#!wget 'https://www.ncei.noaa.gov/access/services/data/v1?dataset=daily-summaries&stations=USW00023174,USW00012960,USW00003017,USW00094846,USW00013874,USW00023234,USW00003927,USW00023183,USW00013881&dataTypes=AWND,PRCP,SNOW,SNWD,TAVG,TMIN,TMAX&startDate=2014-01-01&endDate=2018-12-31' -O /home/ec2-user/SageMaker/project/data/daily-summaries.csv

Impor data cuaca yang disiapkan untuk kode bandara di set data. Gunakan stasiun dan bandara berikut untuk analisis. Buat kolom baru bernama *bandara* yang memetakan stasiun cuaca ke nama bandara.

In [ ]:
weather = pd.read_csv('/home/ec2-user/SageMaker/project/data/daily-summaries.csv')
station = ['USW00023174','USW00012960','USW00003017','USW00094846','USW00013874','USW00023234','USW00003927','USW00023183','USW00013881'] 
airports = ['LAX', 'IAH', 'DEN', 'ORD', 'ATL', 'SFO', 'DFW', 'PHX', 'CLT']

### Map weather stations to airport code
station_map = {s:a for s,a in zip(station, airports)}
weather['airport'] = weather['STATION'].map(station_map)

Dari kolom **TANGGAL**, buat kolom lain yang disebut *BULAN*.

In [ ]:
weather['MONTH'] = weather['DATE'].apply(lambda x: x.split('-')[1])
weather.head()

### Output sampel
```
  STATION     DATE      AWND PRCP SNOW SNWD TAVG TMAX  TMIN airport MONTH
0 USW00023174 2014-01-01 16   0   NaN  NaN 131.0 178.0 78.0  LAX    01
1 USW00023174 2014-01-02 22   0   NaN  NaN 159.0 256.0 100.0 LAX    01
2 USW00023174 2014-01-03 17   0   NaN  NaN 140.0 178.0 83.0  LAX    01
3 USW00023174 2014-01-04 18   0   NaN  NaN 136.0 183.0 100.0 LAX    01
4 USW00023174 2014-01-05 18   0   NaN  NaN 151.0 244.0 83.0  LAX    01
```

Analisis dan tangani kolom **SNOW** dan **SNWD** untuk nilai yang hilang dengan menggunakan `fillna()`. Untuk memeriksa nilai yang hilang di semua kolom, gunakan fungsi `isna()`.

In [ ]:
weather.SNOW.fillna(0, inplace=True)
weather.SNWD.fillna(0, inplace=True)
weather.isna().sum()

**Pertanyaan**: Cetak indeks baris yang tidak memiliki nilai untuk *TAVG*, *TMAX*, *TMIN*.

**Petunjuk**: Untuk menemukan baris yang hilang, gunakan fungsi `isna()`. Kemudian, untuk mendapatkan indeks, gunakan daftar pada variabel *idx*.

In [ ]:
idx = np.array([i for i in range(len(weather))])
TAVG_idx = idx[weather.TAVG.isna()] 
TMAX_idx = # Enter your code here 
TMIN_idx = # Enter your code here 
TAVG_idx

### Output sampel

```
array([ 3956,  3957,  3958,  3959,  3960,  3961,  3962,  3963,  3964,
        3965,  3966,  3967,  3968,  3969,  3970,  3971,  3972,  3973,
        3974,  3975,  3976,  3977,  3978,  3979,  3980,  3981,  3982,
        3983,  3984,  3985,  4017,  4018,  4019,  4020,  4021,  4022,
        4023,  4024,  4025,  4026,  4027,  4028,  4029,  4030,  4031,
        4032,  4033,  4034,  4035,  4036,  4037,  4038,  4039,  4040,
        4041,  4042,  4043,  4044,  4045,  4046,  4047, 13420])
```

Anda dapat mengganti nilai *TAVG*, *TMAX*, dan *TMIN* yang hilang dengan nilai rata-rata untuk stasiun atau bandara tertentu. Karena baris *TAVG_idx* yang berurutan hilang, penggantian dengan nilai sebelumnya tidak dapat dilakukan. Sebagai gantinya, gantilah dengan mean. Gunakan fungsi `groupby` untuk agregat variabel dengan nilai mean.

**Petunjuk:** Grup oleh `MONTH` dan `STATION`.

In [ ]:
weather_impute = weather.groupby([<CODE>]).agg({'TAVG':'mean','TMAX':'mean', 'TMIN':'mean' }).reset_index()# Enter your code here
weather_impute.head(2)

Gabungkan data mean dengan data cuaca.

In [ ]:

weather = pd.merge(weather, weather_impute,  how='left', left_on=['MONTH','STATION'], right_on = ['MONTH','STATION'])\
.rename(columns = {'TAVG_y':'TAVG_AVG',
                   'TMAX_y':'TMAX_AVG', 
                   'TMIN_y':'TMIN_AVG',
                   'TAVG_x':'TAVG',
                   'TMAX_x':'TMAX', 
                   'TMIN_x':'TMIN'})

Periksa kembali nilai yang hilang.

In [ ]:
weather.TAVG[TAVG_idx] = weather.TAVG_AVG[TAVG_idx]
weather.TMAX[TMAX_idx] = weather.TMAX_AVG[TMAX_idx]
weather.TMIN[TMIN_idx] = weather.TMIN_AVG[TMIN_idx]
weather.isna().sum()

Jatuhkan `STATION,MONTH,TAVG_AVG,TMAX_AVG,TMIN_AVG,TMAX,TMIN,SNWD` dari data set.

In [ ]:
weather.drop(columns=['STATION','MONTH','TAVG_AVG', 'TMAX_AVG', 'TMIN_AVG', 'TMAX' ,'TMIN', 'SNWD'],inplace=True)

Tambahkan kondisi cuaca asal dan tujuan ke set data.

In [ ]:
### Add origin weather conditions
data_orig = pd.merge(data_orig, weather,  how='left', left_on=['FlightDate','Origin'], right_on = ['DATE','airport'])\
.rename(columns = {'AWND':'AWND_O','PRCP':'PRCP_O', 'TAVG':'TAVG_O', 'SNOW': 'SNOW_O'})\
.drop(columns=['DATE','airport'])

### Add destination weather conditions
data_orig = pd.merge(data_orig, weather,  how='left', left_on=['FlightDate','Dest'], right_on = ['DATE','airport'])\
.rename(columns = {'AWND':'AWND_D','PRCP':'PRCP_D', 'TAVG':'TAVG_D', 'SNOW': 'SNOW_D'})\
.drop(columns=['DATE','airport'])

**Catatan**: Sebaiknya, selalu periksa nulls atau NAs setelah bergabung.

In [ ]:
sum(data.isna().any())

In [ ]:
data_orig.columns

Konversi data kategorikal menjadi data numerik dengan menggunakan pengkodean one-hot.

In [ ]:
data = data_orig.copy()
data = data[['is_delay', 'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 
       'Reporting_Airline', 'Origin', 'Dest','Distance','DepHourofDay','is_holiday', 'AWND_O', 'PRCP_O',
       'TAVG_O', 'AWND_D', 'PRCP_D', 'TAVG_D', 'SNOW_O', 'SNOW_D']]


categorical_columns  = ['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 
       'Reporting_Airline', 'Origin', 'Dest', 'is_holiday']
for c in categorical_columns:
    data[c] = data[c].astype('category')

In [ ]:
data_dummies = pd.get_dummies(data[['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'Reporting_Airline', 'Origin', 'Dest', 'is_holiday']], drop_first=True)
data = pd.concat([data, data_dummies], axis = 1)
data.drop(categorical_columns,axis=1, inplace=True)

Periksa kolom baru.

In [ ]:
data.shape

In [ ]:
data.columns

### Output sampel

```
Index(['Distance', 'DepHourofDay', 'is_delay', 'AWND_O', 'PRCP_O', 'TAVG_O',
       'AWND_D', 'PRCP_D', 'TAVG_D', 'SNOW_O', 'SNOW_D', 'Year_2015',
       'Year_2016', 'Year_2017', 'Year_2018', 'Quarter_2', 'Quarter_3',
       'Quarter_4', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
       'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12',
       'DayofMonth_2', 'DayofMonth_3', 'DayofMonth_4', 'DayofMonth_5',
       'DayofMonth_6', 'DayofMonth_7', 'DayofMonth_8', 'DayofMonth_9',
       'DayofMonth_10', 'DayofMonth_11', 'DayofMonth_12', 'DayofMonth_13',
       'DayofMonth_14', 'DayofMonth_15', 'DayofMonth_16', 'DayofMonth_17',
       'DayofMonth_18', 'DayofMonth_19', 'DayofMonth_20', 'DayofMonth_21',
       'DayofMonth_22', 'DayofMonth_23', 'DayofMonth_24', 'DayofMonth_25',
       'DayofMonth_26', 'DayofMonth_27', 'DayofMonth_28', 'DayofMonth_29',
       'DayofMonth_30', 'DayofMonth_31', 'DayOfWeek_2', 'DayOfWeek_3',
       'DayOfWeek_4', 'DayOfWeek_5', 'DayOfWeek_6', 'DayOfWeek_7',
       'Reporting_Airline_DL', 'Reporting_Airline_OO', 'Reporting_Airline_UA',
       'Reporting_Airline_WN', 'Origin_CLT', 'Origin_DEN', 'Origin_DFW',
       'Origin_IAH', 'Origin_LAX', 'Origin_ORD', 'Origin_PHX', 'Origin_SFO',
       'Dest_CLT', 'Dest_DEN', 'Dest_DFW', 'Dest_IAH', 'Dest_LAX', 'Dest_ORD',
       'Dest_PHX', 'Dest_SFO', 'is_holiday_1'],
      dtype='object')
```

Ganti nama kolom **is_delay** menjadi *target* kembali. Gunakan kode yang sama dengan yang Anda gunakan sebelumnya.

In [ ]:
data.rename(columns = {<CODE>:<CODE>}, inplace=True )# Enter your code here

Buat set pelatihan lagi.

**Petunjuk:** Gunakan `split_data` fungsi yang Anda tentukan (dan digunakan) sebelumnya.

In [ ]:
# Enter your code here

### Pengklasifikasi dasar baru

Sekarang, lihat apakah fitur baru ini menambahkan kekuatan prediktif ke model.

In [ ]:
# Instantiate the LinearLearner estimator object
classifier_estimator2 = # Enter your code here

### Kode sampel

```
num_classes = len(pd.unique(train_labels)) 
classifier_estimator2 = sagemaker.LinearLearner(role=sagemaker.get_execution_role(),
                                               instance_count=1,
                                               instance_type='ml.m4.xlarge',
                                               predictor_type='binary_classifier',
                                               binary_classifier_model_selection_criteria = 'cross_entropy_loss')
```

In [ ]:
train_records = classifier_estimator2.record_set(train.values[:, 1:].astype(np.float32), train.values[:, 0].astype(np.float32), channel='train')
val_records = classifier_estimator2.record_set(validate.values[:, 1:].astype(np.float32), validate.values[:, 0].astype(np.float32), channel='validation')
test_records = classifier_estimator2.record_set(test.values[:, 1:].astype(np.float32), test.values[:, 0].astype(np.float32), channel='test')

Melatih model Anda dengan menggunakan tiga set data yang baru saja Anda buat.

In [ ]:
# Enter your code here

Lakukan prediksi batch dengan menggunakan model yang baru dilatih.

In [ ]:
# Enter your code here

Plot matriks kekeliruan.

In [ ]:
# Enter your code here

Plot kurva ROC.

In [ ]:
# Enter your code here

Model linier hanya menunjukkan sedikit peningkatan performa. Coba model ansambel berbasis pohon, yang disebut *XGBoost*, dengan Amazon SageMaker.

### Coba model XGBoost

Lakukan langkah-langkah berikut:  

1. Gunakan variabel set pelatihan dan simpan sebagai file CSV: train.csv, validation.csv dan test.csv.
2. Menyimpan nama bucket dalam variabel. Nama bucket Amazon S3 disediakan di sebelah kiri instruksi lab.  
a. `bucket = <LabBucketName>`  
b. `prefix = 'flight-xgb'`  
3. Gunakan AWS SDK for Python (Boto3) untuk mengunggah model ke bucket.    

In [ ]:
bucket='c218151a5506212l17131779t1w715054373660-labbucket-ntokzfktykpe'
prefix='flight-xgb'
train_file='flight_train.csv'
test_file='flight_test.csv'
validate_file='flight_validate.csv'
whole_file='flight.csv'
s3_resource = boto3.Session().resource('s3')

def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, header=False, index=False )
    s3_resource.Bucket(bucket).Object(os.path.join(prefix, folder, filename)).put(Body=csv_buffer.getvalue())

upload_s3_csv(train_file, 'train', train)
upload_s3_csv(test_file, 'test', test)
upload_s3_csv(validate_file, 'validate', validate)

Gunakan fungsi `sagemaker.inputs.TrainingInput` guna membuat `record_set` untuk set data pelatihan dan validasi.

In [ ]:
train_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/train/".format(bucket,prefix,train_file),
    content_type='text/csv')

validate_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/validate/".format(bucket,prefix,validate_file),
    content_type='text/csv')

data_channels = {'train': train_channel, 'validation': validate_channel}

In [ ]:
from sagemaker.image_uris import retrieve
container = retrieve('xgboost',boto3.Session().region_name,'1.0-1')

In [ ]:
sess = sagemaker.Session()
s3_output_location="s3://{}/{}/output/".format(bucket,prefix)

xgb = sagemaker.estimator.Estimator(container,
                                    role = sagemaker.get_execution_role(), 
                                    instance_count=1, 
                                    instance_type=instance_type,
                                    output_path=s3_output_location,
                                    sagemaker_session=sess)
xgb.set_hyperparameters(max_depth=5,
                        eta=0.2,
                        gamma=4,
                        min_child_weight=6,
                        subsample=0.8,
                        silent=0,
                        objective='binary:logistic',
                        eval_metric = "auc", 
                        num_round=100)

xgb.fit(inputs=data_channels)

Gunakan transformator batch untuk model baru Anda, dan evaluasi model pada set data pengujian.

In [ ]:
batch_X = test.iloc[:,1:];
batch_X_file='batch-in.csv'
upload_s3_csv(batch_X_file, 'batch-in', batch_X)

In [ ]:
batch_output = "s3://{}/{}/batch-out/".format(bucket,prefix)
batch_input = "s3://{}/{}/batch-in/{}".format(bucket,prefix,batch_X_file)

xgb_transformer = xgb.transformer(instance_count=1,
                                       instance_type=instance_type,
                                       strategy='MultiRecord',
                                       assemble_with='Line',
                                       output_path=batch_output)

xgb_transformer.transform(data=batch_input,
                         data_type='S3Prefix',
                         content_type='text/csv',
                         split_type='Line')
xgb_transformer.wait()

Dapatkan target yang diprediksi dan label uji.

In [ ]:
s3 = boto3.client('s3')
obj = s3.get_object(Bucket=bucket, Key="{}/batch-out/{}".format(prefix,'batch-in.csv.out'))
target_predicted = pd.read_csv(io.BytesIO(obj['Body'].read()),',',names=['target'])
test_labels = test.iloc[:,0]

Hitung nilai yang diprediksi berdasarkan ambang batas yang ditetapkan.

**Catatan:** Target yang diprediksi akan menjadi skor, yang harus dikonversi ke kelas biner.

In [ ]:
print(target_predicted.head())

def binary_convert(x):
    threshold = 0.55
    if x > threshold:
        return 1
    else:
        return 0

target_predicted['target'] = target_predicted['target'].apply(binary_convert)

test_labels = test.iloc[:,0]

print(target_predicted.head())

Plot matriks kekeliruan untuk `target_predicted` dan `test_labels` Anda.

In [ ]:
# Enter your code here

Plot grafik ROC:

In [ ]:
# Enter your code here

### Cobalah ambang batas yang berbeda

**Pertanyaan**: Berdasarkan seberapa baik model menangani set pengujian, apa yang dapat Anda simpulkan?

In [ ]:
#Enter your answer here

### Optimasi hiperparameter (HPO)

In [ ]:
from sagemaker.tuner import IntegerParameter, CategoricalParameter, ContinuousParameter, HyperparameterTuner

### You can spin up multiple instances to do hyperparameter optimization in parallel

xgb = sagemaker.estimator.Estimator(container,
                                    role=sagemaker.get_execution_role(), 
                                    instance_count= 1, # make sure you have a limit set for these instances
                                    instance_type=instance_type, 
                                    output_path='s3://{}/{}/output'.format(bucket, prefix),
                                    sagemaker_session=sess)

xgb.set_hyperparameters(eval_metric='auc',
                        objective='binary:logistic',
                        num_round=100,
                        rate_drop=0.3,
                        tweedie_variance_power=1.4)

hyperparameter_ranges = {'alpha': ContinuousParameter(0, 1000, scaling_type='Linear'),
                         'eta': ContinuousParameter(0.1, 0.5, scaling_type='Linear'),
                         'min_child_weight': ContinuousParameter(3, 10, scaling_type='Linear'),
                         'subsample': ContinuousParameter(0.5, 1),
                         'num_round': IntegerParameter(10,150)}

objective_metric_name = 'validation:auc'

tuner = HyperparameterTuner(xgb,
                            objective_metric_name,
                            hyperparameter_ranges,
                            max_jobs=10, # Set this to 10 or above depending upon budget and available time.
                            max_parallel_jobs=1)

In [ ]:
tuner.fit(inputs=data_channels)
tuner.wait()

<i class="fas fa-exclamation-triangle" style="color:red"></i> Tunggu sampai tugas pelatihan selesai. Waktu yang diperlukan mungkin 25-30 menit.

**Untuk memantau tugas optimasi hiperparameter:**  

1. Di AWS Management Console, pada menu **Layanan**, pilih **Amazon SageMaker**.  
2. Pilih **Pelatihan > Tugas penyetelan hiperparameter**.
3. Anda dapat memeriksa status setiap tugas penyetelan hiperparameter, nilai metrik objektifnya, dan log nya.  

Periksa apakah tugas berhasil diselesaikan.

In [ ]:
boto3.client('sagemaker').describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuner.latest_tuning_job.job_name)['HyperParameterTuningJobStatus']

Tugas penyetelan hiperparameter akan memiliki model yang bekerja paling baik. Anda bisa mendapatkan informasi tentang model itu dari tugas penyetelan.

In [ ]:
sage_client = boto3.Session().client('sagemaker')
tuning_job_name = tuner.latest_tuning_job.job_name
print(f'tuning job name:{tuning_job_name}')
tuning_job_result = sage_client.describe_hyper_parameter_tuning_job(HyperParameterTuningJobName=tuning_job_name)
best_training_job = tuning_job_result['BestTrainingJob']
best_training_job_name = best_training_job['TrainingJobName']
print(f"best training job: {best_training_job_name}")

best_estimator = tuner.best_estimator()

tuner_df = sagemaker.HyperparameterTuningJobAnalytics(tuning_job_name).dataframe()
tuner_df.head()

Gunakan estimator `best_estimator` dan latih estimator tersebut dengan menggunakan data. 

**Tips:** Lihat fungsi fit estimator XGBoost sebelumnya.

In [ ]:
# Enter your code here'

Gunakan transformator batch untuk model baru Anda, dan evaluasi model pada set data pengujian.

In [ ]:
batch_output = "s3://{}/{}/batch-out/".format(bucket,prefix)
batch_input = "s3://{}/{}/batch-in/{}".format(bucket,prefix,batch_X_file)

xgb_transformer = best_estimator.transformer(instance_count=1,
                                       instance_type=instance_type,
                                       strategy='MultiRecord',
                                       assemble_with='Line',
                                       output_path=batch_output)

xgb_transformer.transform(data=batch_input,
                         data_type='S3Prefix',
                         content_type='text/csv',
                         split_type='Line')
xgb_transformer.wait()

In [ ]:
s3 = boto3.client('s3')
obj = s3.get_object(Bucket=bucket, Key="{}/batch-out/{}".format(prefix,'batch-in.csv.out'))
target_predicted = pd.read_csv(io.BytesIO(obj['Body'].read()),',',names=['target'])
test_labels = test.iloc[:,0]

Dapatkan target yang diprediksi dan label uji.

In [ ]:
print(target_predicted.head())

def binary_convert(x):
    threshold = 0.55
    if x > threshold:
        return 1
    else:
        return 0

target_predicted['target'] = target_predicted['target'].apply(binary_convert)

test_labels = test.iloc[:,0]

print(target_predicted.head())

Plot matriks kekeliruan untuk `target_predicted` dan `test_labels` Anda.

In [ ]:
# Enter your code here

Plot grafik ROC:

In [ ]:
# Enter your code here

**Pertanyaan**: Coba berbagai hiperparameter dan rentang hiperparameter. Apakah perubahan ini memperbaiki model?

## Kesimpulan

Anda sekarang telah melakukan iterasi melalui pelatihan dan evaluasi model Anda setidaknya beberapa kali. Saatnya untuk menyelesaikan proyek ini dan merenungkan:

- Apa yang Anda pelajari 
- Jenis langkah apa saja yang mungkin Anda ambil untuk maju (dengan asumsi Anda memiliki lebih banyak waktu)

Gunakan sel berikut untuk menjawab beberapa pertanyaan ini dan pertanyaan lain yang relevan:

1. Apakah performa model Anda memenuhi tujuan bisnis Anda? Jika tidak, apa saja hal yang ingin Anda lakukan dengan cara berbeda jika Anda memiliki lebih banyak waktu untuk menyetel?
2. Seberapa besar peningkatan yang dihasilkan oleh model Anda saat Anda membuat perubahan pada set data, fitur, dan hiperparameter? Jenis teknik apa saja yang Anda terapkan selama proyek ini, dan yang menghasilkan peningkatan terbesar dalam model Anda?
3. Apa saja tantangan terbesar yang Anda temui selama proyek ini?
4. Apakah Anda memiliki pertanyaan yang belum terjawab tentang aspek alur yang tidak masuk akal bagi Anda?
5. Apa tiga hal terpenting yang Anda pelajari tentang machine learning saat mengerjakan proyek ini?

#### <span style="color: blue;">Presentasi proyek: Pastikan Anda juga meringkas jawaban Anda atas pertanyaan-pertanyaan ini dalam presentasi proyek Anda. Gabungkan semua catatan Anda untuk presentasi proyek Anda dan bersiaplah untuk menyajikan temuan Anda ke kelas. </span>

In [ ]:
# Write your answers here